In [2]:
# ============================================================
# 請求書.xlsx 作成スクリプト
# ポイント:
#   ① datetimeモジュールで今日の日付を取得
#   ② リスト・タプルで会社情報・商品データを管理
#   ③ for文で繰り返し処理
#   ④ enumerateでindex付きループ
# ============================================================

import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

# ============================================================
# ① datetimeモジュール: 今日の日付を YYYY/MM/DD 形式で取得
# ============================================================
today = datetime.date.today()
today_str = today.strftime("%Y/%m/%d")

# ============================================================
# ② リスト・タプルでデータ定義
# ============================================================

# 会社情報（タプル: 固定データ）
company_info = (
    "株式会社ABC",
    "〒101-0022　東京都千代田区神田練塀町300",
    "TEL：03-1234-5678　FAX：03-1234-5678",
    "担当者名：鈴木一郎 様",
)

# 請求番号・日付（タプル）
invoice_meta = ("No.", "0001", "日付", today_str)

# 明細ヘッダー（タプル）
header_cols = ("商品名", "数量", "単価", "金額")

# 商品データ（リスト）各要素は (商品名, 数量, 単価) のタプル
items = [
    ("商品A", 2, 10000),
    ("商品B", 1, 15000),
]

# 集計ラベル（リスト）
summary_labels = ["合計", "消費税", "税込合計"]

# ============================================================
# Workbook・シート作成
# ============================================================
wb = Workbook()
ws = wb.active
ws.title = "Sheet1"

FONT_NAME = "游ゴシック"

# ③ for文: 全列幅を13に設定
col_letters = ["A", "B", "C", "D", "E", "F", "G", "H"]
for col_letter in col_letters:
    ws.column_dimensions[col_letter].width = 13.0

# ============================================================
# ヘルパー関数
# ============================================================
def set_cell(ws, coord, value, size=11, h_align=None, v_align="center"):
    cell = ws[coord]
    cell.value = value
    cell.font = Font(name=FONT_NAME, size=size)
    cell.alignment = Alignment(horizontal=h_align, vertical=v_align)

def set_num_cell(ws, coord, value, size=11, v_align="center"):
    """数値セル: 値を直接設定して確実に表示させる"""
    cell = ws[coord]
    cell.value = value
    cell.font = Font(name=FONT_NAME, size=size)
    cell.alignment = Alignment(vertical=v_align)
    cell.number_format = '#,##0'

# ============================================================
# セルへのデータ書き込み
# ============================================================

# --- タイトル ---
set_cell(ws, "B2", "請求書")

# --- ③ for文 + ④ enumerate: 会社情報をB4〜B7に書き込み ---
start_row = 4
for i, info in enumerate(company_info):       # ④ enumerate → i がindex
    row = start_row + i                       # indexで行番号を動的に決定
    set_cell(ws, f"B{row}", info)

# --- 請求番号・日付 ---
set_cell(ws, "F3", invoice_meta[0])   # No.  → F3
set_cell(ws, "G3", invoice_meta[1])   # 0001 → G3
set_cell(ws, "F4", invoice_meta[2])   # 日付 → F4
set_cell(ws, "G4", invoice_meta[3])   # 日付値 → G4

# --- 明細ヘッダー行10 ---
# ③ for文 + ④ enumerate でB〜E列にヘッダーを書き込み
header_start_col = 2
for idx, label in enumerate(header_cols):     # ④ enumerate → idx がindex
    col_letter = get_column_letter(header_start_col + idx)
    set_cell(ws, f"{col_letter}10", label)

# --- 明細データ行11〜 ---
# ③ for文 + ④ enumerate で商品を1行ずつ書き込み
item_start_row = 11
subtotal = 0

for idx, item in enumerate(items):            # ④ enumerate → idx がindex
    row = item_start_row + idx                # indexで行番号を動的に計算
    name, qty, price = item                   # タプルのアンパック
    amount = qty * price                      # 金額をPythonで計算
    subtotal += amount

    set_cell(ws, f"B{row}", name)
    set_num_cell(ws, f"C{row}", qty)
    set_num_cell(ws, f"D{row}", price)
    set_num_cell(ws, f"E{row}", amount)       # 金額を数値で直接設定

# 小計行（行13）
subtotal_row = item_start_row + len(items)
set_num_cell(ws, f"E{subtotal_row}", subtotal)

# --- 集計セクション (行15〜17) ---
summary_start_row = 15
tax_rate = 0.1
tax = int(subtotal * tax_rate)
total_with_tax = subtotal + tax

# 集計値リスト（リスト）
summary_values = [subtotal, tax, total_with_tax]

# ③ for文 + ④ enumerate で合計・消費税・税込合計を書き込み
for idx, label in enumerate(summary_labels):  # ④ enumerate → idx がindex
    row = summary_start_row + idx             # indexで行番号を動的に計算
    set_cell(ws, f"B{row}", label)
    set_num_cell(ws, f"E{row}", summary_values[idx])  # 数値を直接設定

# ============================================================
# 保存 & ダウンロード（Google Colab対応）
# ============================================================
OUTPUT_PATH = "/content/請求書.xlsx"
wb.save(OUTPUT_PATH)
print(f"保存完了: {OUTPUT_PATH}")
print(f"発行日付: {today_str}")
print(f"小計: {subtotal:,}円 / 消費税: {tax:,}円 / 税込合計: {total_with_tax:,}円")

# Google Colabの場合、自動でダウンロードダイアログを表示
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
    print("ダウンロードを開始しました。")
except ImportError:
    print("※ Colab以外の環境では上記パスから直接ファイルを取得してください。")


保存完了: /content/請求書.xlsx
発行日付: 2026/04/25
小計: 35,000円 / 消費税: 3,500円 / 税込合計: 38,500円


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ダウンロードを開始しました。
